In [2]:
import pandas as pd
import requests

print("Docker Jupyter is working!")
print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)

Docker Jupyter is working!
Pandas: 3.0.5
Requests: 2.34.2


In [3]:
# FSA endpoint for local authorities
url = "https://api.ratings.food.gov.uk/Authorities"

headers = {
    "x-api-version": "2"
}

# Send request
response = requests.get(
    url,
    headers=headers
)

# Check status
print(response.status_code)

200


In [4]:
data = response.json()

print(data.keys())

dict_keys(['authorities', 'meta', 'links'])


In [5]:
data

{'authorities': [{'LocalAuthorityId': 197,
   'LocalAuthorityIdCode': '760',
   'Name': 'Aberdeen City',
   'FriendlyName': 'aberdeen-city',
   'Url': 'http://www.aberdeencity.gov.uk',
   'SchemeUrl': '',
   'Email': 'commercial@aberdeencity.gov.uk',
   'RegionName': 'Scotland',
   'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS760en-GB.xml',
   'FileNameWelsh': None,
   'EstablishmentCount': 2202,
   'CreationDate': '2010-08-17T15:30:24.87',
   'LastPublishedDate': '2026-08-29T00:34:26.217',
   'SchemeType': 2,
   'links': [{'rel': 'self',
     'href': 'https://api.ratings.food.gov.uk/authorities/197'}]},
  {'LocalAuthorityId': 198,
   'LocalAuthorityIdCode': '761',
   'Name': 'Aberdeenshire',
   'FriendlyName': 'aberdeenshire',
   'Url': 'http://www.aberdeenshire.gov.uk/',
   'SchemeUrl': '',
   'Email': 'environmental@aberdeenshire.gov.uk',
   'RegionName': 'Scotland',
   'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS761en-GB.xml',
   'FileNameWelsh': None,


In [6]:
# Convert the authorities list into a DataFrame
authorities_df = pd.DataFrame(data["authorities"])

# Show the first 5 rows
authorities_df.head()

,LocalAuthorityId,LocalAuthorityIdCode,Name,FriendlyName,Url,SchemeUrl,Email,RegionName,FileName,FileNameWelsh,EstablishmentCount,CreationDate,LastPublishedDate,SchemeType,links
0,197,760,Aberdeen City,aberdeen-city,http://www.aberdeencity.gov.uk,,commercial@aberdeencity.gov.uk,Scotland,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,NaN,2202,2010-08-17T15:30:24.87,2026-08-29T00:34:26.217,2,"[{'rel': 'self', 'href': 'https://api.ratings...."
1,198,761,Aberdeenshire,aberdeenshire,http://www.aberdeenshire.gov.uk/,,environmental@aberdeenshire.gov.uk,Scotland,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,NaN,2783,2010-08-17T15:30:24.87,2026-08-28T00:34:25.673,2,"[{'rel': 'self', 'href': 'https://api.ratings...."
2,277,323,Adur,adur,http://www.adur-worthing.gov.uk,,environmental.health@adur-worthing.gov.uk,South East,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,NaN,463,2010-08-17T15:30:24.87,2026-08-29T00:35:49.917,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
3,48,062,Amber Valley,amber-valley,http://www.ambervalley.gov.uk,,envhealth@ambervalley.gov.uk,East Midlands,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,NaN,1078,2010-08-17T15:30:24.87,2026-08-29T00:30:53.3,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
4,334,551,Anglesey,anglesey,http://www.ynysmon.gov.uk,,iechydyramgylchedd@ynysmon.gov.uk,Wales,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,763,2010-08-17T15:30:24.87,2026-08-21T00:43:32.523,1,"[{'rel': 'self', 'href': 'https://api.ratings...."


In [7]:
authorities_df.shape

(363, 15)

In [8]:
authorities_df.columns

Index(['LocalAuthorityId', 'LocalAuthorityIdCode', 'Name', 'FriendlyName',
       'Url', 'SchemeUrl', 'Email', 'RegionName', 'FileName', 'FileNameWelsh',
       'EstablishmentCount', 'CreationDate', 'LastPublishedDate', 'SchemeType',
       'links'],
      dtype='str')

In [9]:
authorities_df[
    authorities_df["Name"].str.contains("Westminster", case=False, na=False)
][["LocalAuthorityId", "Name", "RegionName", "EstablishmentCount"]]

,LocalAuthorityId,Name,RegionName,EstablishmentCount
346,120,Westminster,London,5722


In [23]:
# FSA food establishments endpoint
url = "https://api.ratings.food.gov.uk/Establishments"

# API version
headers = {
    "x-api-version": "2"
}

 # Fetch a development sample of low-rated FHRS businesses
params = {
    "schemeTypeKey": "FHRS",
    "ratingKey": "2",
    "ratingOperatorKey": "LessThanOrEqual",
    "pageNumber": 1,
    "pageSize": 100
}

# Send request
response = requests.get(
    url,
    headers=headers,
    params=params
)

# Check whether it worked
print(response.status_code)

200


In [24]:
data = response.json()

establishments = data["establishments"]

df_low_rated = pd.DataFrame(establishments)

df_low_rated["RatingValue"].value_counts()

RatingValue
2    100
Name: count, dtype: int64

In [25]:
# Ratings we want in the development sample
ratings = ["0", "1", "2", "3", "4", "5"]

all_samples = []

for rating in ratings:

    params = {
        "schemeTypeKey": "FHRS",
        "ratingKey": rating,
        "pageNumber": 1,
        "pageSize": 50
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    data = response.json()

    sample = pd.DataFrame(data["establishments"])

    all_samples.append(sample)

# Combine all rating groups
df_dev = pd.concat(all_samples, ignore_index=True)

# Check what we received
df_dev["RatingValue"].value_counts().sort_index()

RatingValue
0    50
1    50
2    50
3    50
4    50
5    50
Name: count, dtype: int64

In [26]:
# Save the balanced development sample
df_dev.to_csv(
    "data/raw/fsa_development_sample.csv",
    index=False
)

print("Saved:", df_dev.shape)

Saved: (300, 25)


In [27]:
import json

# Save the balanced development sample in the same JSON structure
# expected by the PostgreSQL loader
development_data = {
    "establishments": df_dev.to_dict(orient="records")
}

with open("data/raw/fsa_development_sample.json", "w") as file:
    json.dump(development_data, file, indent=2)

print("Saved development JSON:", len(development_data["establishments"]))

Saved development JSON: 300


In [12]:
# Convert the API response from JSON
fsa_data = response.json()

# See the main sections returned
print(fsa_data.keys())

dict_keys(['establishments', 'meta', 'links'])


In [13]:
# Look at the first food business
fsa_data["establishments"][0]

{'AddressLine1': '360 Regents Park Road London',
 'AddressLine2': '',
 'AddressLine3': '',
 'AddressLine4': '',
 'BusinessName': '&Market',
 'BusinessType': 'Retailers - other',
 'BusinessTypeID': 4613,
 'ChangesByServerID': 0,
 'Distance': None,
 'FHRSID': 1635494,
 'LocalAuthorityBusinessID': '23/00263/COMM',
 'LocalAuthorityCode': '502',
 'LocalAuthorityEmailAddress': 'FoodSafety@barnet.gov.uk',
 'LocalAuthorityName': 'Barnet',
 'LocalAuthorityWebSite': 'http://www.barnet.gov.uk/',
 'NewRatingPending': False,
 'Phone': '',
 'PostCode': 'N3 2LJ',
 'RatingDate': '2026-04-14T00:00:00',
 'RatingKey': 'fhrs_2_en-gb',
 'RatingValue': '2',
 'RightToReply': '',
 'SchemeType': 'FHRS',
 'geocode': {'longitude': '-0.19403', 'latitude': '51.600817'},
 'scores': {'Hygiene': 0, 'Structural': 15, 'ConfidenceInManagement': 10}}

In [14]:
# Convert the list of food businesses into a DataFrame
establishments_df = pd.DataFrame(fsa_data["establishments"])

# Check the size
print(establishments_df.shape)

# Show the first 5 rows
establishments_df.head()

(100, 25)


,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,Distance,FHRSID,...,NewRatingPending,Phone,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,geocode,scores
0,360 Regents Park Road London,,,,&Market,Retailers - other,4613,0,None,1635494,...,False,,N3 2LJ,2026-04-14T00:00:00,fhrs_2_en-gb,2,,FHRS,"{'longitude': '-0.19403', 'latitude': '51.6008...","{'Hygiene': 0, 'Structural': 15, 'ConfidenceIn..."
1,5-7 Blenheim Parade,Uxbridge Road,,Uxbridge,(TemporailyClosed)Chummy Yummy Chinese Restaur...,Restaurant/Cafe/Canteen,1,0,None,1496743,...,False,,UB10 0LX,2024-02-01T00:00:00,fhrs_2_en-gb,2,,FHRS,"{'longitude': '-0.4458', 'latitude': '51.529624'}","{'Hygiene': 10, 'Structural': 15, 'ConfidenceI..."
2,327 Southbury Road,ENFIELD,,,1 Stop Halal T/A Shazan Select Count,Retailers - other,4613,0,None,1007170,...,False,,EN1 1TW,2025-03-05T00:00:00,fhrs_2_en-gb,2,,FHRS,"{'longitude': '-0.057127', 'latitude': '51.647...","{'Hygiene': 15, 'Structural': 5, 'ConfidenceIn..."
3,1 Foots Cray High Street,Sidcup,Kent,,10.7.2026Cray's Food Centre,Retailers - other,4613,0,None,1618235,...,False,,DA14 5HJ,2026-07-10T00:00:00,fhrs_2_en-gb,2,,FHRS,"{'longitude': '0.118671', 'latitude': '51.4185...","{'Hygiene': 15, 'Structural': 15, 'ConfidenceI..."
4,25 Essex Street,Birmingham,,,100 Degrees,Restaurant/Cafe/Canteen,1,0,None,1875540,...,False,,B5 4TR,2025-11-10T00:00:00,fhrs_2_en-gb,2,,FHRS,"{'longitude': '-1.898752', 'latitude': '52.473...","{'Hygiene': 15, 'Structural': 10, 'ConfidenceI..."


In [15]:
establishments_df.columns

Index(['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4',
       'BusinessName', 'BusinessType', 'BusinessTypeID', 'ChangesByServerID',
       'Distance', 'FHRSID', 'LocalAuthorityBusinessID', 'LocalAuthorityCode',
       'LocalAuthorityEmailAddress', 'LocalAuthorityName',
       'LocalAuthorityWebSite', 'NewRatingPending', 'Phone', 'PostCode',
       'RatingDate', 'RatingKey', 'RatingValue', 'RightToReply', 'SchemeType',
       'geocode', 'scores'],
      dtype='str')

In [16]:
establishments_df["geocode"].head()

0    {'longitude': '-0.19403', 'latitude': '51.6008...
1    {'longitude': '-0.4458', 'latitude': '51.529624'}
2    {'longitude': '-0.057127', 'latitude': '51.647...
3    {'longitude': '0.118671', 'latitude': '51.4185...
4    {'longitude': '-1.898752', 'latitude': '52.473...
Name: geocode, dtype: object

In [17]:
establishments_df["scores"].head()

0    {'Hygiene': 0, 'Structural': 15, 'ConfidenceIn...
1    {'Hygiene': 10, 'Structural': 15, 'ConfidenceI...
2    {'Hygiene': 15, 'Structural': 5, 'ConfidenceIn...
3    {'Hygiene': 15, 'Structural': 15, 'ConfidenceI...
4    {'Hygiene': 15, 'Structural': 10, 'ConfidenceI...
Name: scores, dtype: object

In [18]:
# Turn the geocode dictionary into normal columns
geocode_df = pd.json_normalize(establishments_df["geocode"])

# Show the first 5 rows
geocode_df.head()

,longitude,latitude
0,-0.19403,51.600817
1,-0.4458,51.529624
2,-0.057127,51.647487
3,0.118671,51.418592
4,-1.898752,52.4737396


In [19]:
# Turn the inspection scores dictionary into normal columns
scores_df = pd.json_normalize(establishments_df["scores"])

# Show the first 5 rows
scores_df.head()

,Hygiene,Structural,ConfidenceInManagement
0,0.0,15.0,10.0
1,10.0,15.0,10.0
2,15.0,5.0,10.0
3,15.0,15.0,10.0
4,15.0,10.0,10.0


In [20]:
# Add latitude and longitude to the main DataFrame
establishments_df["latitude"] = geocode_df["latitude"]
establishments_df["longitude"] = geocode_df["longitude"]

# Add inspection scores for analysis only
establishments_df["Hygiene"] = scores_df["Hygiene"]
establishments_df["Structural"] = scores_df["Structural"]
establishments_df["ConfidenceInManagement"] = scores_df["ConfidenceInManagement"]

# Check the result
establishments_df.head()

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,Distance,FHRSID,...,RatingValue,RightToReply,SchemeType,geocode,scores,latitude,longitude,Hygiene,Structural,ConfidenceInManagement
0,360 Regents Park Road London,,,,&Market,Retailers - other,4613,0,None,1635494,...,2,,FHRS,"{'longitude': '-0.19403', 'latitude': '51.6008...","{'Hygiene': 0, 'Structural': 15, 'ConfidenceIn...",51.600817,-0.19403,0.0,15.0,10.0
1,5-7 Blenheim Parade,Uxbridge Road,,Uxbridge,(TemporailyClosed)Chummy Yummy Chinese Restaur...,Restaurant/Cafe/Canteen,1,0,None,1496743,...,2,,FHRS,"{'longitude': '-0.4458', 'latitude': '51.529624'}","{'Hygiene': 10, 'Structural': 15, 'ConfidenceI...",51.529624,-0.4458,10.0,15.0,10.0
2,327 Southbury Road,ENFIELD,,,1 Stop Halal T/A Shazan Select Count,Retailers - other,4613,0,None,1007170,...,2,,FHRS,"{'longitude': '-0.057127', 'latitude': '51.647...","{'Hygiene': 15, 'Structural': 5, 'ConfidenceIn...",51.647487,-0.057127,15.0,5.0,10.0
3,1 Foots Cray High Street,Sidcup,Kent,,10.7.2026Cray's Food Centre,Retailers - other,4613,0,None,1618235,...,2,,FHRS,"{'longitude': '0.118671', 'latitude': '51.4185...","{'Hygiene': 15, 'Structural': 15, 'ConfidenceI...",51.418592,0.118671,15.0,15.0,10.0
4,25 Essex Street,Birmingham,,,100 Degrees,Restaurant/Cafe/Canteen,1,0,None,1875540,...,2,,FHRS,"{'longitude': '-1.898752', 'latitude': '52.473...","{'Hygiene': 15, 'Structural': 10, 'ConfidenceI...",52.4737396,-1.898752,15.0,10.0,10.0


In [21]:
import json

# Save the original API response exactly as we received it
with open("data/raw/fsa_westminster_sample.json", "w") as file:
    json.dump(fsa_data, file, indent=4)

In [22]:
import os

# Check that the raw JSON file exists
print(os.path.exists("data/raw/fsa_westminster_sample.json"))

# Check how many establishments we saved
print(len(fsa_data["establishments"]))

True
100
